# 03 · Monitor a LoRA training run

This notebook shows how a run started by `src.train_lora` (from notebook 02 or the CLI) is progressing.
It **only reads files** in `runs/<run_name>/`, so it is safe to use while training is running:

| File | Written | Used for |
|---|---|---|
| `metrics.jsonl` | one row every `steps_per_report` iterations (`kind=train`) and every `steps_per_eval` (`kind=val`) | all curves |
| `config.yaml` | at start | total iterations, report/eval intervals |
| `run_info.json` | at start, rewritten at the end | `finished` / `best` once done |
| `PID` | at start, detached runs only | is the process alive? |
| `train.log` | detached runs only | last log lines and crash detection |

It **does not import MLX or load the model**, so it takes no GPU memory away from training.

There is no accuracy during training, only losses. **Train loss** is measured on the batches being trained on.
**Val loss** is measured on a fixed held-out subset of `valid.jsonl` and is what selects `best/`.
Span F1 and JSON validity come from the eval step after training.

In [ ]:
import json, os, re, time
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import yaml
from IPython.display import clear_output, display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RUNS = ROOT / "runs"

def latest_run() -> str:
    """The run whose metrics.jsonl changed most recently."""
    files = sorted(RUNS.glob("*/metrics.jsonl"), key=lambda p: p.stat().st_mtime)
    return files[-1].parent.name

print("runs with metrics (newest first):")
for p in sorted(RUNS.glob("*/metrics.jsonl"), key=lambda p: -p.stat().st_mtime):
    print(f"  {p.parent.name:32s} last write {time.strftime('%H:%M:%S', time.localtime(p.stat().st_mtime))}")

## Pick the run

`RUN_NAME = None` follows whichever run wrote metrics most recently. Set a name to pin one, e.g. `"qwen3-4b-lora-r16-v1"`.

In [ ]:
RUN_NAME = None          # None = most recently active run
REFRESH_S = 60           # live view refresh interval
SMOOTH = 0.3             # EMA weight for the smoothed train-loss line (1.0 = no smoothing)

run = RUN_NAME or latest_run()
RUN_DIR = RUNS / run
print("monitoring", RUN_DIR.relative_to(ROOT))

## Helpers

- `load(run_dir)` reads the files above into a train table and a val table.
- `status(...)` works out whether the process is alive, how far it has got, and an ETA. The ETA uses the average
  seconds per iteration since the first report, so time spent on periodic validation is already included.
- `plot(...)` draws four panels: losses (log scale), learning rate, throughput, and peak memory.

In [ ]:
def load(run_dir: Path):
    cfg = yaml.safe_load((run_dir / "config.yaml").read_text())
    rows = [json.loads(l) for l in (run_dir / "metrics.jsonl").read_text().splitlines() if l.strip()]
    df = pd.DataFrame(rows)
    tr = df[df.kind == "train"].reset_index(drop=True)
    va = df[df.kind == "val"].reset_index(drop=True)
    info = json.loads((run_dir / "run_info.json").read_text()) if (run_dir / "run_info.json").exists() else {}
    return cfg, tr, va, info


def process_alive(run_dir: Path) -> bool | None:
    """True/False from the PID file; None if the run has no PID file (e.g. started from the notebook cell)."""
    pid_file = run_dir / "PID"
    if not pid_file.exists():
        return None
    try:
        os.kill(int(pid_file.read_text().strip()), 0)
        return True
    except (ProcessLookupError, ValueError):
        return False
    except PermissionError:
        return True


def status(run_dir: Path, cfg, tr, va, info) -> dict:
    total = int(cfg["iters"])
    it = int(tr.iteration.iloc[-1]) if len(tr) else 0
    last_write = (run_dir / "metrics.jsonl").stat().st_mtime
    alive = process_alive(run_dir)
    if "finished" in info:
        state = "finished"
    elif alive or (alive is None and time.time() - last_write < 15 * 60):
        state = "running"
    else:
        state = "STOPPED before finishing (check train.log)"
    s = {"run": run_dir.name, "state": state, "iteration": f"{it} / {total} ({100 * it / total:.1f}%)"}
    if len(tr) >= 2:
        sec_per_it = (tr.wall_s.iloc[-1] - tr.wall_s.iloc[0]) / (tr.iteration.iloc[-1] - tr.iteration.iloc[0])
        s["sec / iteration"] = round(sec_per_it, 1)
        if state == "running":
            remaining = (total - it) * sec_per_it
            s["ETA"] = f"{remaining / 3600:.1f} h  (~{time.strftime('%H:%M', time.localtime(time.time() + remaining))})"
    if len(tr):
        last = tr.iloc[-1]
        s["train loss (last)"] = round(last.train_loss, 4)
        s["learning rate"] = f"{last.learning_rate:.2e}"
        s["tokens / s"] = round(last.tokens_per_second, 1)
        s["peak memory GB"] = round(last.peak_memory, 1)
        s["elapsed"] = f"{last.wall_s / 3600:.2f} h"
    if len(va):
        b = va.loc[va.val_loss.idxmin()]
        s["val loss (base, it 0)"] = round(va.val_loss.iloc[0], 4)
        s["val loss (best)"] = f"{b.val_loss:.4f} at it {int(b.iteration)}"
    s["last metrics write"] = time.strftime("%H:%M:%S", time.localtime(last_write))
    return s


def plot(cfg, tr, va, run_name: str):
    fig, ax = plt.subplots(2, 2, figsize=(13, 8))
    a = ax[0, 0]
    if len(tr):
        a.plot(tr.iteration, tr.train_loss, color="tab:blue", alpha=0.3, label="train (raw)")
        a.plot(tr.iteration, tr.train_loss.ewm(alpha=SMOOTH).mean(), color="tab:blue", label="train (smoothed)")
    if len(va):
        a.plot(va.iteration, va.val_loss, "o-", color="tab:orange", label="val")
        b = va.loc[va.val_loss.idxmin()]
        if b.iteration > 0:
            a.plot(b.iteration, b.val_loss, "*", ms=16, color="tab:red", label=f"best val {b.val_loss:.4f}")
    a.set_yscale("log"); a.set_title("loss (log scale)"); a.set_xlabel("iteration"); a.legend(); a.grid(alpha=0.3)

    for a, col, title in [(ax[0, 1], "learning_rate", "learning rate"),
                          (ax[1, 0], "tokens_per_second", "throughput (trained tokens / s)"),
                          (ax[1, 1], "peak_memory", "peak memory (GB)")]:
        if len(tr):
            a.plot(tr.iteration, tr[col], ".-")
        a.set_title(title); a.set_xlabel("iteration"); a.grid(alpha=0.3)
    for a in ax.flat:
        a.set_xlim(0, cfg["iters"])
    fig.suptitle(run_name)
    fig.tight_layout()
    return fig


def tail_log(run_dir: Path, n: int = 8):
    log = run_dir / "train.log"
    if not log.exists():
        return
    text = re.sub(r"\x1b\[[0-9;]*m", "", log.read_text(errors="replace"))  # drop mlx-lm's colour codes
    bad = ("Traceback", "Error", "Killed")
    # keep our own [train]/[val]/[best] lines plus anything that looks like a failure
    lines = [l for l in text.splitlines() if l.startswith(("[", "lengths", "done")) or any(k in l for k in bad)]
    if any(k in l for l in lines for k in bad):
        print("!! train.log contains an error, see below")
    print("\n".join(["--- train.log (last lines) ---", *lines[-n:]]))


def snapshot(run_dir: Path = None):
    run_dir = run_dir or RUN_DIR
    cfg, tr, va, info = load(run_dir)
    st = status(run_dir, cfg, tr, va, info)
    display(pd.Series(st, name="").to_frame())
    plt.show(plot(cfg, tr, va, run_dir.name))
    if len(va):
        display(va[["iteration", "val_loss", "val_time", "wall_s"]].round(4).set_index("iteration").T)
    tail_log(run_dir)
    return st

## Snapshot

Run this cell any time for the current state. It does not wait.

In [ ]:
_ = snapshot()

## Live view

Redraws every `REFRESH_S` seconds until the run finishes or stops. Press **Interrupt** (■) to stop watching.
Stopping this cell does not affect training.

Because this kernel is busy while the cell runs, use it from this notebook, not from the training notebook.

In [ ]:
try:
    while True:
        clear_output(wait=True)
        st = snapshot()
        print(f"\nrefreshed {time.strftime('%H:%M:%S')}; next in {REFRESH_S}s (Interrupt to stop watching)")
        if st["state"] != "running":
            print("run is no longer running; live view stopped")
            break
        time.sleep(REFRESH_S)
except KeyboardInterrupt:
    print("stopped watching (training continues)")

## How to read the curves

- **Train loss** is noisy because each report averages only a few batches. Look at the smoothed line.
- **Val loss** is the signal. If it rises while train loss keeps falling, the model is overfitting.
  `best/` keeps the weights from the lowest val loss, so a late rise does not hurt the saved adapter.
- **Learning rate** should warm up over the first few iterations, then follow the cosine decay.
- **Throughput** drops if something else is using the GPU, for example a second training or generation process.
- **Peak memory** is a running maximum, so it only steps up, each time a longer batch than any before arrives.
  Steps are normal; what matters is that it stays well below the machine's memory (48 GB here).